# Setup

In [1]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import gurobipy as grb
from gurobipy import quicksum
from gurobipy import Model
from gurobipy import *
import random
from matplotlib import pyplot as plt
from matplotlib import colors
import numpy as np
import itertools
import timeit

In [2]:
import numpy as np

np.std([20001.023677257006,20000.40630608599]) # Change stdev to std

np.float64(0.3086855855071917)

In [3]:
np.median([20001.023677257006,20000.40630608599])

np.float64(20000.7149916715)

# Functions

### RP Function

In [4]:
def RP(Cost_Factor,Fixed_Cost,Recourse_Cost,Capacity,Demand_Scenarios_List,Demand_Probabilities_List,J,seed):

    Scenarios = 8
    np.random.seed(seed)
    All_Scenarios = np.array([np.random.choice(Demand_Scenarios_List[i],Scenarios,p=Demand_Probabilities_List[i]) for i in J]).T
    All_Probabilities = np.array([1.0/Scenarios for i in range(Scenarios)])


    ### 1. Sets
    I = [i for i in range(len(x_dc))] #Warehouses
    J = [j for j in range(len(demand_areas))] #Demand Zones
    Ω = [ω for ω in range(Scenarios)] #Scenarios
    IΩ = [(i,ω) for i in I for ω in Ω] #Recourse - Scenarios
    IJ = [(i,j) for i in I for j in J] #Arcs
    IJΩ = [(i,j,ω) for i in I for j in J for ω in Ω] #Arcs - Scenarios
    JΩ = [(j,ω) for j in J for ω in Ω] #Recourse - Scenarios

    ### 2. Parameters
    q_i_j = {(i,j): eucledian(x_dc[i],y_dc[i],x_demand[j],y_demand[j])*Cost_Factor for i in I for j in J}
    c_i = {(i): Fixed_Cost for i in I}
    m_i = {(i): Capacity for i in I}
    z_j = {(j): Recourse_Cost for j in J}
    π_ω = {(ω): All_Probabilities[ω] for ω in Ω}
    d_j_ω = dict()
    d_j = dict()
    for j in J:
        demand_scenario = All_Scenarios.T[j]
        for ω in range(Scenarios):
            d_j_ω[j,ω] = demand_scenario[ω]
    d_j = {(j): Demand_Averages_List[j] for j in J}

    ### 3. Model and Decision Variables
    RP = Model('RP(Ω)')
    RP.params.logtoconsole=0
    x_i = RP.addVars(I, vtype=GRB.BINARY, lb=0.0, ub=1.0)
    y_i_j_ω = RP.addVars(IJΩ, vtype=GRB.CONTINUOUS, lb=0)
    h_j_ω = RP.addVars(JΩ, vtype=GRB.CONTINUOUS, lb=0)

    ### 4. Objective Function
    RP.modelSense = GRB.MINIMIZE
    first_stage = quicksum([c_i[i]*x_i[i] for i in I])
    second_stage_transport = quicksum([q_i_j[i,j]*y_i_j_ω[i,j,ω]*π_ω[ω] for i in I for j in J for ω in Ω])
    second_stage_recourse = quicksum([z_j[j]*h_j_ω[j,ω]*π_ω[ω] for j in J for ω in Ω])

    RP.setObjective(first_stage+second_stage_transport+second_stage_recourse,GRB.MINIMIZE)

    ### 5. Constraints
    RP.addConstrs(quicksum([y_i_j_ω[i,j,ω] for i in I]) + h_j_ω[j,ω] >= d_j_ω[j,ω] for j in J for ω in Ω)
    RP.addConstrs(quicksum([y_i_j_ω[i,j,ω] for j in J]) <= m_i[i]*x_i[i] for i in I for ω in Ω)
    RP.update()

    ### 6. Optimize
    RP.Params.MIPFocus = 2
    RP.Params.Method = 3
    RP.optimize()

    return RP.ObjVal

### WS Function

In [5]:
def WS(Cost_Factor,Fixed_Cost,Recourse_Cost,Capacity,Demand_Scenarios_List,Demand_Probabilities_List,J,seed):

    Scenarios = 8
    np.random.seed(seed)
    All_Scenarios = np.array([np.random.choice(Demand_Scenarios_List[i],Scenarios,p=Demand_Probabilities_List[i]) for i in J]).T
    All_Probabilities = np.array([1.0/Scenarios for i in range(Scenarios)])

    ### 1. Sets
    I = [i for i in range(len(x_dc))] #Warehouses
    J = [j for j in range(len(demand_areas))] #Demand Zones
    Ω = [ω for ω in range(Scenarios)] #Scenarios
    IΩ = [(i,ω) for i in I for ω in Ω] #Recourse - Scenarios
    IJ = [(i,j) for i in I for j in J] #Arcs
    IJΩ = [(i,j,ω) for i in I for j in J for ω in Ω] #Arcs - Scenarios
    JΩ = [(j,ω) for j in J for ω in Ω] #Recourse - Scenarios

    ### 2. Parameters
    q_i_j = {(i,j): eucledian(x_dc[i],y_dc[i],x_demand[j],y_demand[j])*Cost_Factor for i in I for j in J}
    c_i = {(i): Fixed_Cost for i in I}
    m_i = {(i): Capacity for i in I}
    z_j = {(j): Recourse_Cost for j in J}
    π_ω = {(ω): All_Probabilities[ω] for ω in Ω}
    d_j_ω = dict()
    d_j = dict()
    for j in J:
        demand_scenario = All_Scenarios.T[j]
        for ω in range(Scenarios):
            d_j_ω[j,ω] = demand_scenario[ω]
    d_j = {(j): Demand_Averages_List[j] for j in J}

    ### 3. Model and Decision Variables
    WS = Model('WS(Ω)')
    x_i_ω = WS.addVars(IΩ, vtype=GRB.BINARY, lb=0.0, ub=1.0)
    y_i_j_ω = WS.addVars(IJΩ, vtype=GRB.CONTINUOUS, lb=0)
    h_j_ω = WS.addVars(JΩ, vtype=GRB.CONTINUOUS, lb=0)
    WS.params.logtoconsole=0

    ### 4. Objective Function
    WS.modelSense = GRB.MINIMIZE
    first_stage = quicksum([c_i[i]*x_i_ω[i,ω]*π_ω[ω] for i in I for ω in Ω])
    second_stage_transport = quicksum([q_i_j[i,j]*y_i_j_ω[i,j,ω]*π_ω[ω] for i in I for j in J for ω in Ω])
    second_stage_recourse = quicksum([z_j[j]*h_j_ω[j,ω]*π_ω[ω] for j in J for ω in Ω])

    WS.setObjective(first_stage+second_stage_transport+second_stage_recourse,GRB.MINIMIZE)

    ### 5. Constraints
    WS.addConstrs(quicksum([y_i_j_ω[i,j,ω] for i in I]) + h_j_ω[j,ω] >= d_j_ω[j,ω] for j in J for ω in Ω)
    WS.addConstrs(quicksum([y_i_j_ω[i,j,ω] for j in J]) <= m_i[i]*x_i_ω[i,ω] for i in I for ω in Ω)
    WS.update()

    #WS.Params.MIPGap = 0.05    # 5%
    #WS.Params.TimeLimit = 300  # 5 minutes
    WS.Params.MIPFocus = 2
    WS.Params.Method = 3
    WS.optimize()

    return WS.ObjVal



### WS($\phi$) Function

In [7]:
def WS_Phi(IGS, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost, seed=None):
    # Scenario Generation (Embedded)
    J = [j for j in range(len(demand_areas))]  # Demand Zones
    IGS = list(IGS)
    J_Stoch = [x for x in J if x not in IGS]

    Scenarios_Ω = 8
    Scenarios_Φ = 8

    np.random.seed(seed)
    All_Scenarios = np.array([np.random.choice(Demand_Scenarios_List[i], Scenarios_Ω, p=Demand_Probabilities_List[i]) for i in J_Stoch]).T
    IGS_All_Scenarios = np.array([np.random.choice(Demand_Scenarios_List[i], Scenarios_Φ, p=Demand_Probabilities_List[i]) for i in IGS]).T
    All_Probabilities = np.array([1.0 / Scenarios_Ω for i in range(Scenarios_Ω)])
    IGS_All_Probabilities = np.array([1.0 / Scenarios_Φ for i in range(Scenarios_Φ)])


    # WSPhi Logic (Embedded)
    I = [i for i in range(len(x_dc))]  # Warehouses
    Ω = [ω for ω in range(Scenarios_Ω)]  # Scenarios
    Φ = [ϕ for ϕ in range(Scenarios_Φ)]  # Scenarios
    IΦ = [(i, ϕ) for i in I for ϕ in Φ]  # Recourse - Scenarios ϕ
    IJΩΦ = [(i, j, ω, ϕ) for i in I for j in J for ω in Ω for ϕ in Φ]  # Arcs - Scenarios
    JΩΦ = [(j, ω, ϕ) for j in J for ω in Ω for ϕ in Φ]  # Recourse - Scenarios

    q_i_j = {(i, j): eucledian(x_dc[i], y_dc[i], x_demand[j], y_demand[j]) * Cost_Factor for i in I for j in J}
    c_i = {(i): Fixed_Cost for i in I}
    m_i = {(i): Capacity for i in I}
    z_j = {(j): Recourse_Cost for j in J}
    π_ω = {(ω): All_Probabilities[ω] for ω in Ω}
    π_ϕ = {(ϕ): IGS_All_Probabilities[ϕ] for ϕ in Φ}

    d_j_ω = dict()
    d_j_ϕ = dict()

    for i in range(len(J_Stoch)):
        demand_scenario = All_Scenarios.T[i]
        for ω in range(Scenarios_Ω):
            d_j_ω[J_Stoch[i], ω] = demand_scenario[ω]

    for i in range(len(IGS)):
        demand_scenario = IGS_All_Scenarios.T[i]
        for ϕ in range(Scenarios_Φ):
            d_j_ϕ[IGS[i], ϕ] = demand_scenario[ϕ]

    WSΦ = Model('WS(Φ)')
    WSΦ.Params.logtoconsole = 0
    WSΦ.Params.OutputFlag = 0
    WSΦ.Params.MIPFocus = 2
    WSΦ.Params.Method = 3
    x_i_ϕ = WSΦ.addVars(IΦ, vtype=GRB.BINARY, lb=0.0, ub=1.0)
    y_i_j_ω_ϕ = WSΦ.addVars(IJΩΦ, vtype=GRB.CONTINUOUS, lb=0)
    h_j_ω_ϕ = WSΦ.addVars(JΩΦ, vtype=GRB.CONTINUOUS, lb=0)

    WSΦ.modelSense = GRB.MINIMIZE
    first_stage = quicksum([c_i[i] * x_i_ϕ[i, ϕ] * π_ϕ[ϕ] for i in I for ϕ in Φ])
    second_stage_transport = quicksum([q_i_j[i, j] * y_i_j_ω_ϕ[i, j, ω, ϕ] * π_ω[ω] * π_ϕ[ϕ] for i in I for j in J for ω in Ω for ϕ in Φ])
    second_stage_recourse = quicksum([z_j[j] * h_j_ω_ϕ[j, ω, ϕ] * π_ω[ω] * π_ϕ[ϕ] for j in J for ω in Ω for ϕ in Φ])

    WSΦ.setObjective(first_stage + second_stage_transport + second_stage_recourse, GRB.MINIMIZE)

    WSΦ.addConstrs(quicksum([y_i_j_ω_ϕ[i, j, ω, ϕ] for i in I]) + h_j_ω_ϕ[j, ω, ϕ] >= d_j_ω[j, ω] for j in J_Stoch for ϕ in Φ for ω in Ω)
    WSΦ.addConstrs(quicksum([y_i_j_ω_ϕ[i, j, ω, ϕ] for i in I]) + h_j_ω_ϕ[j, ω, ϕ] >= d_j_ϕ[j, ϕ] for j in IGS for ϕ in Φ for ω in Ω)
    WSΦ.addConstrs(quicksum([y_i_j_ω_ϕ[i, j, ω, ϕ] for j in J]) <= m_i[i] * x_i_ϕ[i, ϕ] for i in I for ϕ in Φ for ω in Ω)
    WSΦ.update()


    WSΦ.optimize()

    return WSΦ.ObjVal

# Example Usage (Assuming you have the necessary variables defined)
# result = WS_Phi(IGS, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost)
# print(result)

### SAA Functions

In [8]:
def SAA_RP(Cost_Factor,Fixed_Cost,Recourse_Cost,Capacity,Demand_Scenarios_List,Demand_Probabilities_List,G):
  RP_Averages = []
  for seed_i in range(2):
      RPΩ = RP(Cost_Factor,Fixed_Cost,Recourse_Cost,Capacity,Demand_Scenarios_List,Demand_Probabilities_List,G,seed_i)
      RP_Averages.append(RPΩ)

  return np.mean(RP_Averages)

def SAA_WS(Cost_Factor,Fixed_Cost,Recourse_Cost,Capacity,Demand_Scenarios_List,Demand_Probabilities_List,G):
  WS_Averages = []
  for seed_i in range(2):
      WSΩ = WS(Cost_Factor,Fixed_Cost,Recourse_Cost,Capacity,Demand_Scenarios_List,Demand_Probabilities_List,G,seed_i)
      WS_Averages.append(WSΩ)
  return np.mean(WS_Averages)

def SAA_WS_Phi(ϕ, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost):
  WSΦ_Averages = []
  for seed_i in range(2):
      WSΦ_sample = WS_Phi(ϕ, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost, seed_i)
      WSΦ_Averages.append(WSΦ_sample)
  return np.mean(WSΦ_Averages)


### Cost Function CGI

In [9]:
info_cost = 200
def CGI(ϕ):
  len(ϕ)
  return len(ϕ)*info_cost

### ETSCI

In [10]:
def ETSCI_ϕ(ϕ, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost):
  return SAA_WS_Phi(ϕ, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost) + CGI(ϕ)


# 10x10 Case

In [11]:
print('10x10 Case')

10x10 Case


### Case

In [12]:
#Create Blocks
x_blocks = 10
y_blocks = 10
case = x_blocks*y_blocks

#Create Demand
CV = 0.25
mu, sigma = 100, 100*CV # mean and standard deviation
np.random.seed(2022)
s = abs(np.random.normal(mu, sigma, 200))
demand_areas = s[:case].reshape(y_blocks,x_blocks)

#Create Demand Coordinates
demand_coord_x = np.arange(0.5,x_blocks,1).tolist()
x_demand = demand_coord_x*y_blocks
demand_coord_y = np.arange(0.5,y_blocks,1)[::-1].tolist()
#y_demand = demand_coord_y*x_blocks
y_demand = []
for a in demand_coord_y:
    y_demand = y_demand+[a]*x_blocks

#Create DC Coordinates
dc_coord_x = np.arange(1,x_blocks,1).tolist()
dc_coord_y = np.arange(1,y_blocks,1)[::-1].tolist()
x_dc = []
y_dc = []
for a in dc_coord_y:
    for b in dc_coord_x:
        y_dc = y_dc+[a]
        x_dc = x_dc+[b]

x_dc = [2, 8, 5, 2, 8]
y_dc = [8, 8, 5, 2, 2]


#plt.plot([1, 0.5], [1, 1.5], c='y', zorder=0)
plt.show()
demand_areas = demand_areas.flatten()
seed = 10
CV_in_Zone = 0.25
Demand_Scenarios_List = []
Demand_Probabilities_List = []
Demand_Averages_List = []
G = [j for j in range(len(demand_areas))] #Demand Zones
for j in G:
    np.random.seed(seed)
    demand_j = np.random.normal(demand_areas[j],demand_areas[j]*CV_in_Zone,1000)
    demand_j[demand_j<0]=0
    (n, bins) = np.histogram(demand_j, 3)
    demand_prob_j = n/sum(n)
    demand_scen_j = np.array([(bins[i]+bins[i+1])/2.0 for i in range(len(bins)-1)])
    avg_demand_j = demand_prob_j@demand_scen_j
    Demand_Probabilities_List.append(demand_prob_j)
    Demand_Scenarios_List.append(demand_scen_j)
    Demand_Averages_List.append(avg_demand_j)


Cost_Factor = 5
Fixed_Cost = 30000
Recourse_Cost = 500
Capacity = 2500

def eucledian(x_dc,y_dc,x_demand,y_demand): #calculating Euclidean distance
    point1 = np.array((x_dc,y_dc))
    point2 = np.array((x_demand,y_demand))
    dist = np.linalg.norm(point1 - point2)
    return dist #return Euclidean distance


### Initial Computations

In [13]:
RP_Value = SAA_RP(Cost_Factor,Fixed_Cost,Recourse_Cost,Capacity,Demand_Scenarios_List,Demand_Probabilities_List,G)
RP_Value

Set parameter Username
Set parameter LicenseID to value 2764064


Academic license - for non-commercial use only - expires 2027-01-13
Set parameter LogToConsole to value 0
Set parameter LogToConsole to value 0


np.float64(231069.79576440674)

In [14]:
WSG = SAA_WS(Cost_Factor,Fixed_Cost,Recourse_Cost,Capacity,Demand_Scenarios_List,Demand_Probabilities_List,G)
WSG

Set parameter LogToConsole to value 0
Set parameter LogToConsole to value 0


np.float64(222783.75758111218)

In [15]:
EVPI = RP_Value - WSG
EVPI

np.float64(8286.038183294557)

In [16]:
WS_Phi_G = SAA_WS_Phi(G, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost)
WS_Phi_G

Set parameter LogToConsole to value 0
Set parameter LogToConsole to value 0


np.float64(222783.75758111276)

In [ ]:
RP_Value =SAA_WS_Phi([], demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost)
RP_Value

34253.98818244064

In [ ]:
WS_Phi_G = SAA_WS_Phi(G, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost)
WS_Phi_G

32997.03990946954

In [ ]:
ETSCI_G = WS_Phi_G + CGI(G)
ETSCI_G

37997.03990946954

In [ ]:
ETSCI_Em = RP_Value
ETSCI_Em

34253.98818244064

### Approximation Bounds Method

In [ ]:
starttime = timeit.default_timer()
EVPPI_i = dict()
for i in G:
  EVPPI_i[i] = RP_Value - SAA_WS_Phi([i], demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost)
  #print('EVPPI_',i,':',np.round(EVPPI_i[i],0))

ϕ_bounds = []
for i in G:
  #print('Savings = ',CGI([i])-EVPPI_i[i])
  if CGI([i])-EVPPI_i[i] <= 0:
    ϕ_bounds.append(i)
time_bounds1 = timeit.default_timer() - starttime
ETSCI_ϕ_bounds = ETSCI_ϕ(ϕ_bounds, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost)
if ETSCI_ϕ_bounds > ETSCI_Em:
  ϕ_bounds = []
  ETSCI_ϕ_bounds = ETSCI_Em
if ETSCI_ϕ_bounds > ETSCI_G:
  ϕ_bounds = G
  ETSCI_ϕ_bounds = ETSCI_G

print("The time difference for the bounds is :", time_bounds1)
print('ϕ_bounds',ϕ_bounds)
print('ETSCI_ϕ_bounds',ETSCI_ϕ_bounds)

EVPPI_ 0 : -56.0
EVPPI_ 1 : -210.0
EVPPI_ 2 : -18.0
EVPPI_ 3 : -388.0
EVPPI_ 4 : -133.0
EVPPI_ 5 : -104.0
EVPPI_ 6 : -151.0
EVPPI_ 7 : -222.0
EVPPI_ 8 : -227.0
EVPPI_ 9 : -256.0
EVPPI_ 10 : -107.0
EVPPI_ 11 : -5.0
EVPPI_ 12 : -946.0
EVPPI_ 13 : -975.0
EVPPI_ 14 : -973.0
EVPPI_ 15 : -899.0
EVPPI_ 16 : -704.0
EVPPI_ 17 : -658.0
EVPPI_ 18 : -791.0
EVPPI_ 19 : -715.0
EVPPI_ 20 : -371.0
EVPPI_ 21 : -387.0
EVPPI_ 22 : 151.0
EVPPI_ 23 : 1055.0
EVPPI_ 24 : 1059.0
Savings =  255.95075399402413
Savings =  410.1682256208005
Savings =  217.99757822250103
Savings =  587.73084196946
Savings =  333.36125326619367
Savings =  304.2893385506468
Savings =  351.25824671497685
Savings =  421.5276707464218
Savings =  426.61923818964715
Savings =  455.57168048426684
Savings =  306.5857096657419
Savings =  204.7346231802585
Savings =  1145.54821491451
Savings =  1174.948841238118
Savings =  1173.110300520042
Savings =  1099.1395372504194
Savings =  903.8446125742485
Savings =  857.6954228604518
Savings =  991

In [ ]:
starttime = timeit.default_timer()
EVPPI_i = dict()
for i in G:
  EVPPI_i[i] = RP_Value - SAA_WS_Phi([i], demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost)
  #print('EVPPI_',i,':',np.round(EVPPI_i[i],0))

ϕ_bounds = []
for i in G:
  #print('Savings = ',CGI([i])-EVPPI_i[i])
  if CGI([i])-EVPPI_i[i] <= 0:
    ϕ_bounds.append(i)
time_bounds2 = timeit.default_timer() - starttime
ETSCI_ϕ_bounds = ETSCI_ϕ(ϕ_bounds, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost)
if ETSCI_ϕ_bounds > ETSCI_Em:
  ϕ_bounds = []
  ETSCI_ϕ_bounds = ETSCI_Em
if ETSCI_ϕ_bounds > ETSCI_G:
  ϕ_bounds = G
  ETSCI_ϕ_bounds = ETSCI_G
print("The time difference for the bounds is :", time_bounds2)
print('ϕ_bounds',ϕ_bounds)
print('ETSCI_ϕ_bounds',ETSCI_ϕ_bounds)

EVPPI_ 0 : 1233.0
EVPPI_ 1 : 1163.0
EVPPI_ 2 : 771.0
EVPPI_ 3 : 557.0
EVPPI_ 4 : 741.0
EVPPI_ 5 : 880.0
EVPPI_ 6 : 922.0
EVPPI_ 7 : 860.0
EVPPI_ 8 : 864.0
EVPPI_ 9 : 808.0
EVPPI_ 10 : 833.0
EVPPI_ 11 : 834.0
Savings =  -1032.7142943827603
Savings =  -963.2276536321115
Savings =  -571.1253488176044
Savings =  -357.32653220272186
Savings =  -541.4916464250164
Savings =  -680.4884533905606
Savings =  -721.5366209877975
Savings =  -660.3368013479194
Savings =  -664.3341431998597
Savings =  -607.56777437653
Savings =  -632.9847892415382
Savings =  -633.55961889424
The time difference for the bounds is : 35.68630129099995
ϕ_bounds [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
ETSCI_ϕ_bounds 18729.875066245448


In [ ]:
ETSCI_ϕ_bounds = ETSCI_ϕ(ϕ_bounds, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost)
ETSCI_ϕ_bounds

18729.875066245448

### Approximate Projected Subgradient Method

In [ ]:
ETSCI_G = ETSCI_ϕ(G, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost)
def tilde_ETSCI(ϕ):
  return ETSCI_ϕ(list(set(G) - set(ϕ)), demand_areas, Demand_Scenarios_List,
                 Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand,
                 y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost) - ETSCI_G

In [ ]:
T = 3
N = len(G)
R = 2 * np.sqrt(N)
L = np.abs(np.sum([EVPPI_i[i] for i in G])-EVPI)
eta = R / (L * np.sqrt(T))

starttime = timeit.default_timer()
"""
Approximate Algorithm for solving the problem in a reasonable computational time.

Parameters:
- ETSCI: Function that computes ETSCI given a set of nodes.
- G: List of all nodes in the network (Graph).
- T: Number of steps.
- eta: Learning rate.
- phi_initial: Initial set of nodes, defaults to empty set.

Returns:
- Optimal set of nodes `hat_phi` minimizing the objective function.
"""

# Initializations
N = len(G)
phi_0 = set() # Changed phi_0 to an empty set
np.random.seed(12)
varphi = np.random.uniform(0.5, 0.5, N)  # Random initialization in [0,1]^N


for t in range(T):
    #print(t)
    # Flip and Argsort operations to obtain phi_N
    phi_N = list(np.flip(np.argsort(varphi)))
    #print(varphi)
    #print(phi_N)
    kappa = np.zeros(N)

    # Calculate subgradients for each g_l
    for l in range(N):
        kappa[phi_N[l]] = tilde_ETSCI(phi_N[:l+1]) - tilde_ETSCI(phi_N[:l])

    # Update varphi using subgradients
    varphi -= eta * kappa
    varphi = np.clip(varphi, 0, 1)  # Projection onto [0,1]^N

# Obtain the superlevel set
hat_phi_N = list(np.flip(np.argsort(varphi)))
hat_phi_0 = set()
tilde_phi = min(range(N + 1), key=lambda l: tilde_ETSCI(set(hat_phi_N[:l])))

# Compute the solution for ETSCI
hat_phi = set(G) - set(hat_phi_N[:tilde_phi])

J = hat_phi_N
#print(J)
m=tilde_ETSCI([])
S=[]
for i in range(N):
    m_c = tilde_ETSCI(J[:i+1])
    #print(m_c,J[:i+1])
    if m_c<m:
        m=m_c
        S=J[:i+1]
S=list(filter(lambda i:i not in S,G))
S = sorted(S)
time_apsm1 = timeit.default_timer() - starttime
print("The time difference for the approximation algorithm is :", time_apsm1)
ESTCI_apsm = ETSCI_ϕ(S, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost)
print(ESTCI_apsm,S)

The time difference for the approximation algorithm is : 337.54699205299994
17947.11250670424 [0, 3, 5, 6, 9, 10]


In [ ]:
T = 3
N = len(G)
R = 2 * np.sqrt(N)
L = np.abs(np.sum([EVPPI_i[i] for i in G])-EVPI)
eta = R / (L * np.sqrt(T))

starttime = timeit.default_timer()
"""
Approximate Algorithm for solving the problem in a reasonable computational time.

Parameters:
- ETSCI: Function that computes ETSCI given a set of nodes.
- G: List of all nodes in the network (Graph).
- T: Number of steps.
- eta: Learning rate.
- phi_initial: Initial set of nodes, defaults to empty set.

Returns:
- Optimal set of nodes `hat_phi` minimizing the objective function.
"""

# Initializations
N = len(G)
phi_0 = set() # Changed phi_0 to an empty set
np.random.seed(12)
varphi = np.random.uniform(0.5, 0.5, N)  # Random initialization in [0,1]^N


for t in range(T):
    #print(t)
    # Flip and Argsort operations to obtain phi_N
    phi_N = list(np.flip(np.argsort(varphi)))
    #print(varphi)
    #print(phi_N)
    kappa = np.zeros(N)

    # Calculate subgradients for each g_l
    for l in range(N):
        kappa[phi_N[l]] = tilde_ETSCI(phi_N[:l+1]) - tilde_ETSCI(phi_N[:l])

    # Update varphi using subgradients
    varphi -= eta * kappa
    varphi = np.clip(varphi, 0, 1)  # Projection onto [0,1]^N

# Obtain the superlevel set
hat_phi_N = list(np.flip(np.argsort(varphi)))
hat_phi_0 = set()
tilde_phi = min(range(N + 1), key=lambda l: tilde_ETSCI(set(hat_phi_N[:l])))

# Compute the solution for ETSCI
hat_phi = set(G) - set(hat_phi_N[:tilde_phi])
J = hat_phi_N
#print(J)
m=tilde_ETSCI([])
S=[]
for i in range(N):
    m_c = tilde_ETSCI(J[:i+1])
    #print(m_c,J[:i+1])
    if m_c<m:
        m=m_c
        S=J[:i+1]
S=list(filter(lambda i:i not in S,G))
S = sorted(S)
time_apsm2 = timeit.default_timer() - starttime
print("The time difference for the approximation algorithm is :", time_apsm2)
ESTCI_apsm = ETSCI_ϕ(S, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List, x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost)
print(ESTCI_apsm,S)

The time difference for the approximation algorithm is : 339.35362153100004
17947.11250670424 [0, 3, 5, 6, 9, 10]


### Monolithic Exact

In [ ]:
from itertools import chain, combinations
power_set_of_G = [list(subset) for subset in chain.from_iterable(combinations(G, r) for r in range(len(G) + 1))]

In [ ]:
starttime = timeit.default_timer()
ETSCI_min = 10000000
ϕ_min = []
time_limit = 20000  # Set the time limit to 20000 seconds

for i in power_set_of_G:
    # Calculate the ETSCI candidate
    ETSCI_cand = ETSCI_ϕ(i, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List,
                         x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost)

    # If the new candidate is better, update the minimum
    if ETSCI_min > ETSCI_cand:
        ETSCI_min = ETSCI_cand
        ϕ_min = i

    # Check if the time limit has been exceeded
    time_min1 = timeit.default_timer() - starttime
    if time_min1 > time_limit:
        print("Time limit exceeded, stopping the loop.")
        break

# Print the results
print("The time difference for the monolithic exact is:", time_min1)
print(ETSCI_min, ϕ_min)

In [ ]:
starttime = timeit.default_timer()
ETSCI_min = 10000000
ϕ_min = []
time_limit = 20000  # Set the time limit to 20000 seconds

for i in power_set_of_G:
    # Calculate the ETSCI candidate
    ETSCI_cand = ETSCI_ϕ(i, demand_areas, Demand_Scenarios_List, Demand_Probabilities_List, Demand_Averages_List,
                         x_dc, y_dc, x_demand, y_demand, Cost_Factor, Fixed_Cost, Capacity, Recourse_Cost)

    # If the new candidate is better, update the minimum
    if ETSCI_min > ETSCI_cand:
        ETSCI_min = ETSCI_cand
        ϕ_min = i

    # Check if the time limit has been exceeded
    time_min2 = timeit.default_timer() - starttime
    if time_min2 > time_limit:
        print("Time limit exceeded, stopping the loop.")
        break

# Print the results
print("The time difference for the monolithic exact is:", time_min2)
print(ETSCI_min, ϕ_min)


### Reporting Results

In [ ]:
print('Approximation Bounds:')
print('- Average Time:',np.mean([time_bounds1,time_bounds2]))
print('- Standard Deviation Time:',np.std([time_bounds1,time_bounds2]))
print('- Solution:',ϕ_bounds)
print('- Value:',ETSCI_ϕ_bounds)

print('Approximation Algorithm:')
print('- Average Time:',np.mean([time_apsm1,time_apsm2]))
print('- Standard Deviation Time:',np.std([time_apsm1,time_apsm2]))
print('- Solution:',S)
print('- Value:',ESTCI_apsm)

print('Monolithic Exact:')
print('- Average Time:',np.mean([time_min1,time_min2]))
print('- Standard Deviation Time:',np.std([time_min1,time_min2]))
print('- Solution:',ϕ_min)
print('- Value:',ETSCI_min)